## Silver Layer — Flight Data Cleansing & SCD Type 1

This notebook reads raw flight records from Azure SQL Staging, cleans them up, 
builds a stable identity for each flight, detects what has changed between 
loads, and merges the results into a Delta table using SCD Type 1 
(insert new records, update changed records, leave unchanged records alone).

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

### Step 1 — Read raw data from Azure SQL Staging

We connect to Staging.FlightData via JDBC and pull in whatever rows are 
currently there. This is raw, unprocessed data — exactly as ADF loaded it.

In [0]:
jdbc_url = "jdbc:sqlserver://aviation-sql-server.database.windows.net:1433;database=AviationDB;encrypt=true;trustServerCertificate=false;loginTimeout=30;"

connection_properties = {
    "user": "as_aviationdb",
    "password": "flightadmin$123",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

df_staging = spark.read.jdbc(
    url=jdbc_url,
    table="Staging.FlightData",
    properties=connection_properties
)

df_staging.limit(10).display()
print(f"Row count: {df_staging.count()}")

YEAR,MONTH,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,IngestionID,SourceFileName,SourceFileHash,LoadedAt
2026,1,2026-01-02,DL,412,12451,JAX,"Jacksonville, FL",Florida,12953,LGA,"New York, NY",New York,1205,1234,29.00,29.00,1418,1439,21.00,21.00,0.00,null,0.00,133.00,125.00,108.00,834.00,0.00,0.00,0.00,0.00,21.00,1,2026_01.csv,null,2026-08-10T14:30:22.577Z
2026,1,2026-01-02,DL,413,12892,LAX,"Los Angeles, CA",California,13487,MSP,"Minneapolis, MN",Minnesota,1155,1146,-9.00,0.00,1735,1723,-12.00,0.00,0.00,null,0.00,220.00,217.00,190.00,1535.00,null,null,null,null,null,1,2026_01.csv,null,2026-08-10T14:30:22.577Z
2026,1,2026-01-02,DL,414,10397,ATL,"Atlanta, GA",Georgia,11298,DFW,"Dallas/Fort Worth, TX",Texas,1100,1141,41.00,41.00,1231,1301,30.00,30.00,0.00,null,0.00,151.00,140.00,112.00,731.00,30.00,0.00,0.00,0.00,0.00,1,2026_01.csv,null,2026-08-10T14:30:22.577Z
2026,1,2026-01-02,DL,415,14747,SEA,"Seattle, WA",Washington,12478,JFK,"New York, NY",New York,1136,1218,42.00,42.00,2007,2039,32.00,32.00,0.00,null,0.00,331.00,321.00,276.00,2422.00,1.00,0.00,0.00,0.00,31.00,1,2026_01.csv,null,2026-08-10T14:30:22.577Z
2026,1,2026-01-02,DL,416,10299,ANC,"Anchorage, AK",Alaska,10397,ATL,"Atlanta, GA",Georgia,2015,2052,37.00,37.00,718,741,23.00,23.00,0.00,null,0.00,423.00,409.00,365.00,3417.00,23.00,0.00,0.00,0.00,0.00,1,2026_01.csv,null,2026-08-10T14:30:22.577Z
2026,1,2026-01-02,DL,416,10397,ATL,"Atlanta, GA",Georgia,10299,ANC,"Anchorage, AK",Alaska,1515,1513,-2.00,0.00,1857,1853,-4.00,0.00,0.00,null,0.00,462.00,460.00,436.00,3417.00,null,null,null,null,null,1,2026_01.csv,null,2026-08-10T14:30:22.577Z
2026,1,2026-01-02,DL,417,12889,LAS,"Las Vegas, NV",Nevada,10397,ATL,"Atlanta, GA",Georgia,2204,2203,-1.00,0.00,500,502,2.00,2.00,0.00,null,0.00,236.00,239.00,191.00,1747.00,null,null,null,null,null,1,2026_01.csv,null,2026-08-10T14:30:22.577Z
2026,1,2026-01-02,DL,418,10849,BZN,"Bozeman, MT",Montana,10721,BOS,"Boston, MA",Massachusetts,1325,1325,0.00,0.00,1955,1926,-29.00,0.00,0.00,null,0.00,270.00,241.00,214.00,1991.00,null,null,null,null,null,1,2026_01.csv,null,2026-08-10T14:30:22.577Z
2026,1,2026-01-02,DL,419,14747,SEA,"Seattle, WA",Washington,12173,HNL,"Honolulu, HI",Hawaii,1730,1727,-3.00,0.00,2200,2125,-35.00,0.00,0.00,null,0.00,390.00,358.00,338.00,2677.00,null,null,null,null,null,1,2026_01.csv,null,2026-08-10T14:30:22.577Z
2026,1,2026-01-02,DL,420,12892,LAX,"Los Angeles, CA",California,11298,DFW,"Dallas/Fort Worth, TX",Texas,1830,1848,18.00,18.00,2329,2338,9.00,9.00,0.00,null,0.00,179.00,170.00,143.00,1235.00,null,null,null,null,null,1,2026_01.csv,null,2026-08-10T14:30:22.577Z


Row count: 544003


### Step 2 — Clean up CANCELLED and DIVERTED

These arrived from the source as decimal values (0.00 / 1.00), not true 
booleans, because of a type-mismatch issue at the ADF Copy Activity level. 
We convert them to real True/False here.

In [0]:
df_silver = df_staging.withColumn("CANCELLED_BOOL",when(col("CANCELLED")==1.0,True).otherwise(False))\
                    .withColumn("DIVERTED_BOOL",when(col("DIVERTED")==1.0,True).otherwise(False))

In [0]:
df_silver=df_silver.drop("CANCELLED","DIVERTED").withColumnRenamed("CANCELLED_BOOL","CANCELLED")\
    .withColumnRenamed("DIVERTED_BOOL","DIVERTED")

In [0]:
df_silver.select("FL_DATE","OP_UNIQUE_CARRIER","CANCELLED","DIVERTED").limit(10).display()

FL_DATE,OP_UNIQUE_CARRIER,CANCELLED,DIVERTED
2026-01-02,DL,false,false
2026-01-02,DL,false,false
2026-01-02,DL,false,false
2026-01-02,DL,false,false
2026-01-02,DL,false,false
2026-01-02,DL,false,false
2026-01-02,DL,false,false
2026-01-02,DL,false,false
2026-01-02,DL,false,false
2026-01-02,DL,false,false


### Step 3 — Build the Unique flight identity (natural key) and Verify the key is actually unique

BTS data has no ID column, so we build one ourselves. A flight is uniquely 
identified by: the date it flew, the airline, the flight number, where it 
departed from and arrived at, and its scheduled departure time. These six 
values together will never legitimately change for a given flight — they're 
what make it *that specific flight* and not another one.
Before creating FlightKey as an identity, we check whether any two rows 
share the exact same key. Zero results here means the key is safe to use.

In [0]:
key_cols = ["FL_DATE", "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST", "CRS_DEP_TIME"]
dupe_check = df_silver.groupBy(key_cols).count().filter(col("count")>1)

FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,count


In [0]:
dupe_count = dupe_check.count()
print(f"Duplicate key groups: {dupe_count}")
if dupe_count > 0:
    dupe_check.limit(10).display()

Duplicate key groups: 0


In [0]:
df_silver = df_silver.withColumn("FlightKey",concat_ws("||",col("FL_DATE"),col("OP_UNIQUE_CARRIER"),col("OP_CARRIER_FL_NUM")\
    ,col("ORIGIN"),col("DEST"),col("CRS_DEP_TIME")))

df_silver.select("FL_DATE","OP_UNIQUE_CARRIER","OP_CARRIER_FL_NUM","ORIGIN","DEST","CRS_DEP_TIME","FlightKey").limit(10).display()

FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,FlightKey
2026-01-02,DL,412,JAX,LGA,1205,2026-01-02||DL||412||JAX||LGA||1205
2026-01-02,DL,413,LAX,MSP,1155,2026-01-02||DL||413||LAX||MSP||1155
2026-01-02,DL,414,ATL,DFW,1100,2026-01-02||DL||414||ATL||DFW||1100
2026-01-02,DL,415,SEA,JFK,1136,2026-01-02||DL||415||SEA||JFK||1136
2026-01-02,DL,416,ANC,ATL,2015,2026-01-02||DL||416||ANC||ATL||2015
2026-01-02,DL,416,ATL,ANC,1515,2026-01-02||DL||416||ATL||ANC||1515
2026-01-02,DL,417,LAS,ATL,2204,2026-01-02||DL||417||LAS||ATL||2204
2026-01-02,DL,418,BZN,BOS,1325,2026-01-02||DL||418||BZN||BOS||1325
2026-01-02,DL,419,SEA,HNL,1730,2026-01-02||DL||419||SEA||HNL||1730
2026-01-02,DL,420,LAX,DFW,1830,2026-01-02||DL||420||LAX||DFW||1830


### Step 4 — Hash the flight key

FlightKey is a readable but variable-length string. We convert it into a 
fixed-length hash (FlightKeyHash) because hashes are faster to compare and 
join on at scale — this is what MERGE will actually use to match rows.

In [0]:
df_silver = df_silver.withColumn("FlightKeyHash",sha2(col("FlightKey"),256))
df_silver.select("FlightKey","FlightKeyHash").limit(10).display()

FlightKey,FlightKeyHash
2026-01-02||DL||412||JAX||LGA||1205,00b54af791e849e9590d8a06355b81df0269c4845bee120577179913473910ed
2026-01-02||DL||413||LAX||MSP||1155,42a19c0d1195b2a4b316b0fe3f30105ef232039f66ff97ac5a6379fda203b839
2026-01-02||DL||414||ATL||DFW||1100,1502d6359fb809206080d303f10121dbca5610ac9bb4786fb48427d4d4c40ab5
2026-01-02||DL||415||SEA||JFK||1136,9ad727283558ad9f5cf04d2528f8b1c53fdfaa6f0fbc91c7176c35abd85aaac0
2026-01-02||DL||416||ANC||ATL||2015,43d6919f5fd79e2199167fb1dc026b46f5ae331b09e6ee7ceecf35fc05527e28
2026-01-02||DL||416||ATL||ANC||1515,e203eadcf91b4d6aa76d1f26639c28c5e5406f5772917de33b70b7feecb75027
2026-01-02||DL||417||LAS||ATL||2204,e1fd6b53e8549a01d4cb0731977263fde1caccb07a86fbe92894a273ca28b7be
2026-01-02||DL||418||BZN||BOS||1325,3cd86ef9c70dfc4255ac48e743f0e7ef7dba20f99f1a7f9049f51bfa69f96af9
2026-01-02||DL||419||SEA||HNL||1730,b83d726c45722e465e885fb348c21a095687e14ca30b5662850966546e0f794a
2026-01-02||DL||420||LAX||DFW||1830,7ecb8571e3c3db11b6a27b1cf636c2cde0400955f2671b5c366c9cc10b6f9dbe


### Step 5 — Build the change-detection fingerprint

FlightKey identifies *which* flight this is — but we also need to know 
*what we currently know about it*, since delay/cancellation info can change 
between loads (a flight's actual delay gets recorded, a cancellation gets 
flagged later, etc). We combine those "can change" columns into one string. 
Missing (NULL) delay-cause values are explicitly marked as "NULL" text so 
they're never silently confused with a real 0.00 value.

In [0]:
df_silver = df_silver.withColumn("RowChanged",concat_ws("||",
                                                        col("DEP_TIME").cast("string"),
                                                        col("DEP_DELAY").cast("string"),
                                                        col("ARR_TIME").cast("string"),
                                                        col("ARR_DELAY").cast("string"),
                                                        col("CANCELLED").cast("string"),
                                                        col("DIVERTED").cast("string"),
                                                        coalesce(col("CARRIER_DELAY").cast("string"), lit("NULL")),
                                                        coalesce(col("WEATHER_DELAY").cast("string"), lit("NULL")),
                                                        coalesce(col("NAS_DELAY").cast("string"), lit("NULL")),
                                                        coalesce(col("SECURITY_DELAY").cast("string"), lit("NULL")),
                                                        coalesce(col("LATE_AIRCRAFT_DELAY").cast("string"), lit("NULL"))))
df_silver.select("RowChanged").display()

RowChanged
1234||29.00||1439||21.00||false||false||0.00||0.00||0.00||0.00||21.00
1146||-9.00||1723||-12.00||false||false||NULL||NULL||NULL||NULL||NULL
1141||41.00||1301||30.00||false||false||30.00||0.00||0.00||0.00||0.00
1218||42.00||2039||32.00||false||false||1.00||0.00||0.00||0.00||31.00
2052||37.00||741||23.00||false||false||23.00||0.00||0.00||0.00||0.00
1513||-2.00||1853||-4.00||false||false||NULL||NULL||NULL||NULL||NULL
2203||-1.00||502||2.00||false||false||NULL||NULL||NULL||NULL||NULL
1325||0.00||1926||-29.00||false||false||NULL||NULL||NULL||NULL||NULL
1727||-3.00||2125||-35.00||false||false||NULL||NULL||NULL||NULL||NULL
1848||18.00||2338||9.00||false||false||NULL||NULL||NULL||NULL||NULL


### Step 6 — Hash the change fingerprint

Same reasoning as Step 4 — RowChangeHash is a fixed-length summary of all 
11 "can change" columns. MERGE will compare this single value instead of 
checking eleven columns individually.

In [0]:
df_silver = df_silver.withColumn("RowChangedHash",sha2(col("RowChanged"),256))
df_silver.select("RowChanged","RowChangedHash").limit(10).display()

RowChanged,RowChangedHash
1234||29.00||1439||21.00||false||false||0.00||0.00||0.00||0.00||21.00,86006f721f0b6f2f89cd4552323e31dd722a0fd5ab09309bf1c5dc85fb81412d
1146||-9.00||1723||-12.00||false||false||NULL||NULL||NULL||NULL||NULL,7833ce3bf6e83392c28f7589a25c4ff84e93822265d41befef45cddb2f53a3dc
1141||41.00||1301||30.00||false||false||30.00||0.00||0.00||0.00||0.00,0c018f94b5fdd5e9dd1f2ddd65ad25b7a683863198303daad1e998cdd4861a4a
1218||42.00||2039||32.00||false||false||1.00||0.00||0.00||0.00||31.00,d5f13bf6fa0db6328710bbdefe1b05f009cb7be363fcb7f9e0e43bf9a7247e6b
2052||37.00||741||23.00||false||false||23.00||0.00||0.00||0.00||0.00,fc482f122683060804b6ffad775302178267ea8a71f5af993070612d7acefb3a
1513||-2.00||1853||-4.00||false||false||NULL||NULL||NULL||NULL||NULL,b52819ec1a8daf5edbf4fc34c93afb63c73f81f997e82b13966989a15579a3c2
2203||-1.00||502||2.00||false||false||NULL||NULL||NULL||NULL||NULL,ab959352b7c6884e88e12517791d3fd3f25323de597aaf2f3ab07b2925be1cdf
1325||0.00||1926||-29.00||false||false||NULL||NULL||NULL||NULL||NULL,03ea5307d0a151a29c2af182db0b84934c7f3857f878255222676a7fd95349d6
1727||-3.00||2125||-35.00||false||false||NULL||NULL||NULL||NULL||NULL,3fcafcf7d27b55ea3b6b6fcc3879e9bac0eed2040c11d65149ea13a361091b9b
1848||18.00||2338||9.00||false||false||NULL||NULL||NULL||NULL||NULL,0f21216dc1d4ff47fcf60e0b2336fc281e22c0886e051eeaf92099fd1ec665f0


### Step 7 — Create the table (first run only) or MERGE (every run after)

First run: the Silver table doesn't exist, so we create it directly from 
this batch of data.

Every run after that: the table already has data in it, so we use MERGE 
instead — this is the SCD Type 1 logic:
- A flight we've never seen before → INSERT as a new row
- A flight we've seen, but something changed → UPDATE that row
- A flight we've seen, and nothing changed → do nothing

This makes the notebook safe to re-run any number of times without 
duplicating or destroying data.

In [0]:
table_exist = spark.catalog.tableExists("aviation_ws.silver.flight_data")
if  not table_exist:
    df_silver.write.format("delta").saveAsTable("aviation_ws.silver.flight_data")
    print("Table created : First Run")
else:
    silver_table = DeltaTable.forName(spark,"aviation_ws.silver.flight_data")
    silver_table.alias("target").merge(df_silver.alias("source"),"target.FlightKeyHash=source.FlightKeyHash")\
                                .whenMatchedUpdateAll(condition="target.RowChangedHash != source.RowChangedHash")\
                                .whenNotMatchedInsertAll()\
                                .execute()
                                
    print("MERGE Completed")


MERGE Completed


### Step 8 — Verify

Confirm the table has the expected row count.

In [0]:
%sql
select count(*) as data_count from aviation_ws.silver.flight_data

data_count
544003


In [0]:
df_silver.printSchema()

root
 |-- YEAR: short (nullable = true)
 |-- MONTH: short (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_NM: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: decimal(10,2) (nullable = true)
 |-- DEP_DELAY_NEW: decimal(10,2) (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: decimal(10,2) (nullable = true)
 |-- ARR_DELAY_NEW: decimal(10,2) (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- CRS_